In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!gmx --version

!apt-get remove --purge gromacs


Mounted at /content/drive
/bin/bash: line 1: gmx: command not found
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'gromacs' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
# Download the latest version of GROMACS 2024.3
!wget https://ftp.gromacs.org/gromacs/gromacs-2024.4.tar.gz

# Extract the tarball
!tar xfz gromacs-2024.4.tar.gz
# Navigate to the GROMACS source directory
%cd gromacs-2024.4

# Create a build directory
!mkdir build
%cd build

# Configure the build with CUDA (GPU) support
!cmake .. -DGMX_BUILD_OWN_FFTW=ON -DGMX_GPU=CUDA -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs

# Compile the source code using all available cores
!make -j$(nproc)

# Install GROMACS
!make install


--2026-01-05 16:21:10--  https://ftp.gromacs.org/gromacs/gromacs-2024.4.tar.gz
Resolving ftp.gromacs.org (ftp.gromacs.org)... 130.237.11.165, 2001:6b0:1:1191:216:3eff:fec7:6e30
Connecting to ftp.gromacs.org (ftp.gromacs.org)|130.237.11.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42356162 (40M) [application/x-gzip]
Saving to: ‘gromacs-2024.4.tar.gz’

gromacs-2024.4.tar. 100%[===================>]  40.39M  14.4MB/s    in 2.8s    

2026-01-05 16:21:14 (14.4 MB/s) - ‘gromacs-2024.4.tar.gz’ saved [42356162/42356162]

/content/gromacs-2024.4
/content/gromacs-2024.4/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working C

In [ ]:
# Manually export GROMACS binary to the PATH
import os
os.environ['PATH'] += ":/usr/local/gromacs/bin"



In [ ]:
!gmx --version

In [ ]:
# Install required packages
!apt-get install -y csh
!apt-get update
!apt-get install -y build-essential cmake git libfftw3-dev libgsl-dev

# Install CUDA (if not already installed)
!apt-get install cuda

# Check if GPU is available (optional, but useful for verification)
!nvidia-smi

In [ ]:
%cd "/content/drive/MyDrive/MD/Silicaprotein/Native2"
import os
import subprocess

def run_command(command):
    """Runs a shell command and prints the output."""
    try:
        process = subprocess.run(command, shell=True, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        print(f"COMMAND SUCCESS: {command}")
    except subprocess.CalledProcessError as e:
        print(f"COMMAND FAILED: {command}\nError: {e.stderr}")
        raise

def create_nvt_mdp():
    """Creates a standard nvt.mdp file if it is missing."""
    nvt_content = """
title                   = Protein NVT equilibration
define                  = -DPOSRES  ; position restrain the protein
; Run parameters
integrator              = md        ; leap-frog integrator
nsteps                  = 50000     ; 2 * 50000 = 100 ps
dt                      = 0.002     ; 2 fs
; Output control
nstxout                 = 500       ; save coordinates every 1.0 ps
nstvout                 = 500       ; save velocities every 1.0 ps
nstenergy               = 500       ; save energies every 1.0 ps
nstlog                  = 500       ; update log file every 1.0 ps
; Bond parameters
continuation            = no        ; first dynamics run
constraint_algorithm    = lincs     ; holonomic constraints
constraints             = h-bonds   ; bonds involving H are constrained
lincs_iter              = 1         ; accuracy of LINCS
lincs_order             = 4         ; also related to accuracy
; Nonbonded settings
cutoff-scheme           = Verlet    ; Buffered neighbor searching
ns_type                 = grid      ; search neighboring grid cells
nstlist                 = 10        ; 20 fs, largely irrelevant with Verlet scheme
rcoulomb                = 1.0       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.0       ; short-range van der Waals cutoff (in nm)
DispCorr                = EnerPres  ; account for cut-off vdW scheme
; Electrostatics
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
pme_order               = 4         ; cubic interpolation
fourierspacing          = 0.16      ; grid spacing for FFT
; Temperature coupling is on
tcoupl                  = V-rescale             ; modified Berendsen thermostat
tc-grps                 = Protein Non-Protein   ; two coupling groups - more accurate
tau_t                   = 0.1     0.1           ; time constant, in ps
ref_t                   = 300     300           ; reference temperature, one for each group, in K
; Pressure coupling is off
pcoupl                  = no        ; no pressure coupling in NVT
; Periodic boundary conditions
pbc                     = xyz       ; 3-D PBC
; Velocity generation
gen_vel                 = yes       ; assign velocities from Maxwell distribution
gen_temp                = 300       ; temperature for Maxwell distribution
gen_seed                = -1        ; generate a random seed
    """
    with open("nvt.mdp", "w") as f:
        f.write(nvt_content.strip())
    print("Created nvt.mdp file.")

def run_native_protein_md(pdb_file="AF-E7EUS7-F1-model_v4.pdb", output_name="native_control"):
    print(f"--- Starting Native Protein MD Control Group: {output_name} ---")

    # 1. Generate Topology (pdb2gmx)
    # Using AMBER99SB-ILDN and TIP3P water as a standard control group setup
    # If your protein requires a different FF, change these flags.
    print("\n[Step 1] Generating Topology...")
    run_command(f"gmx pdb2gmx -f {pdb_file} -o {output_name}_processed.gro -water tip3p -ff amber99sb-ildn -ignh")

    # 2. Define Simulation Box
    print("\n[Step 2] Defining Box...")
    run_command(f"gmx editconf -f {output_name}_processed.gro -o {output_name}_newbox.gro -c -d 1.0 -bt cubic")

    # 3. Solvate
    print("\n[Step 3] Solvating...")
    run_command(f"gmx solvate -cp {output_name}_newbox.gro -cs spc216.gro -o {output_name}_solv.gro -p topol.top")

    # 4. Add Ions
    print("\n[Step 4] Adding Ions...")
    # We use minim.mdp to generate the tpr for ion addition (dummy run)
    if not os.path.exists("minim.mdp"):
        raise FileNotFoundError("Error: 'minim.mdp' not found. Please upload it.")

    run_command(f"gmx grompp -f minim.mdp -c {output_name}_solv.gro -p topol.top -o ions.tpr -maxwarn 2")
    # Neutralize concentration with 0.15M NaCl (standard physiological)
    # Select group 13 (SOL) automatically usually works, or pipe input
    run_command(f"echo 13 | gmx genion -s ions.tpr -o {output_name}_solv_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15")

    # 5. Energy Minimization
    print("\n[Step 5] Running Energy Minimization...")
    run_command(f"gmx grompp -f minim.mdp -c {output_name}_solv_ions.gro -p topol.top -o em.tpr")
    run_command(f"gmx mdrun -v -deffnm em")

    # 6. NVT Equilibration
    print("\n[Step 6] Running NVT Equilibration...")
    if not os.path.exists("nvt.mdp"):
        create_nvt_mdp() # Create if missing

    run_command(f"gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr")
    run_command(f"gmx mdrun -v -deffnm nvt")

    # 7. NPT Equilibration
    print("\n[Step 7] Running NPT Equilibration...")
    if not os.path.exists("npt.mdp"):
        raise FileNotFoundError("Error: 'npt.mdp' not found. Please upload it.")

    run_command(f"gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr -maxwarn 2")
    run_command(f"gmx mdrun -v -deffnm npt")

    # 8. Production MD
    print("\n[Step 8] Running Production MD...")
    if not os.path.exists("md.mdp"):
        raise FileNotFoundError("Error: 'md.mdp' not found. Please upload it.")

    run_command(f"gmx grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md_0_1.tpr")
    # Using -nb gpu if you are on Colab with GPU
    run_command(f"gmx mdrun -v -deffnm md_0_1")

    print("\n--- Simulation Complete ---")

# --- Execution ---
# Ensure your uploaded mdp files are in the same folder as this script.
# Replace '1AKI.pdb' with your native protein file if you have one.
if __name__ == "__main__":
    # Optional: Download a test control protein (Lysozyme) if you don't have one
    if not os.path.exists("native.pdb"):
        print("Downloading Lysozyme (1AKI) as example native protein...")
        run_command("wget https://files.rcsb.org/download/1AKI.pdb -O native.pdb")
        run_command("grep -v HETATM native.pdb > native_clean.pdb") # Clean non-protein atoms

    run_native_protein_md(pdb_file="native_clean.pdb", output_name="native_control")